# MD-GRTN — Exact Paper Implementation (Colab T4)

**3 Sequences:** X_RecN (recent, t-12→t) | X_HourN (1hr ago, offset=12) | X_DayN (24hr ago, offset=288)

**Hyperparams:** d_model=96, heads=3, L=3, MGRC=6, batch=64, AdamW lr=1e-3 wd=0.01, Huber loss

**Training:** Phase 1 — pre-train 3 U-Net BackNets (MSE, 200 iters) → freeze MD → Phase 2 — train rest (Huber, 800 iters)

---
**Steps to run:**
1. Runtime → Change runtime type → T4 GPU
2. Upload `PEMS08.npz` or mount Drive
3. Run all cells top to bottom

In [ ]:
import torch
print('PyTorch :', torch.__version__)
print('CUDA    :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU     :', torch.cuda.get_device_name(0))
    print('VRAM    :', round(torch.cuda.get_device_properties(0).total_memory/1e9,1), 'GB')

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
import math

# ══════════════════════════════════════════════════
# CONFIG  — exact paper Section V-B-1
# ══════════════════════════════════════════════════
class Config:
    # --- data ---
    data_path    = "PEMS08.npz"       # ← change if needed
    num_nodes    = 170
    in_features  = 3                   # flow, occupancy, speed
    seq_len      = 12                  # T_h = 12 (paper)
    pred_len     = 12                  # T_f = 12 (paper)
    feature_idx  = 0                   # predict flow

    # --- 3-period offsets (paper: recent / hourly / daily) ---
    # each window is seq_len=12 steps wide, shifted by offset
    offset_rec   = 0                   # X_RecN  : t-12  → t
    offset_hour  = 12                  # X_HourN : t-24  → t-12  (~1 hr ago)
    offset_day   = 288                 # X_DayN  : t-300 → t-288 (~24 hrs ago)

    # --- noise (paper Section V-A: Gaussian μ=0, σ=10 for PEMS) ---
    noise_mean   = 0.0
    noise_std    = 10.0

    # --- diffusion ---
    T_diff       = 10
    beta_start   = 1e-4
    beta_end     = 0.02

    # --- model (paper V-B-1) ---
    d_model      = 96                  # divisible by n_heads=3  →  96/3=32 ✓
    n_heads      = 3
    num_layers   = 3                   # STFormer L=3
    mgrc_layers  = 6                   # MGRC 6 stacked layers
    dropout      = 0.1

    # --- training (paper V-B-1) ---
    batch_size      = 64
    lr              = 1e-3
    weight_decay    = 0.01
    max_iters       = 800              # total main-training iterations
    pretrain_iters  = 200              # MD pre-training iterations
    delta_h         = 1.0             # Huber δ
    train_ratio     = 0.7             # 7:1:2 split
    val_ratio        = 0.1

cfg    = Config()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
assert cfg.d_model % cfg.n_heads == 0,     f"d_model={cfg.d_model} not divisible by n_heads={cfg.n_heads}"
print(f"Device  : {device}")
print(f"d_model : {cfg.d_model}  n_heads : {cfg.n_heads}  → head_dim={cfg.d_model//cfg.n_heads} ✓")
print(f"Offsets : rec={cfg.offset_rec} | hour={cfg.offset_hour} | day={cfg.offset_day} steps")

In [ ]:
# ══════════════════════════════════════════════════
# DATA LOADING
# Paper: Z-score normalisation then Gaussian noise
# added to inputs; labels stay clean (noise-free)
# ══════════════════════════════════════════════════

def load_pems08(cfg):
    raw  = np.load(cfg.data_path, allow_pickle=True)
    print('Keys in npz:', list(raw.keys()))
    data = raw['data'].astype(np.float32)         # (T, N, F)
    T, N, F = data.shape
    print(f'Raw shape   : {data.shape}')

    # Z-score per sensor per feature  (paper V-A)
    mean = data.mean(axis=0, keepdims=True)        # (1,N,F)
    std  = data.std(axis=0,  keepdims=True) + 1e-8
    data_norm = (data - mean) / std               # clean normalised

    # Adjacency matrix from npz
    for key in ('adjacency_matrix','adj_mx','adj','W'):
        if key in raw:
            A = raw[key].astype(np.float32)
            print(f'Adjacency   : loaded from "{key}" shape={A.shape}')
            break
    else:
        print('Adjacency   : not found → using identity')
        A = np.eye(N, dtype=np.float32)

    # Row-normalise A
    A = A / (A.sum(1, keepdims=True) + 1e-8)
    return data_norm, mean, std, A


class MDGRTNDataset(Dataset):
    """
    Returns 7 tensors per sample:
      xr_n, xh_n, xd_n  — NOISY  inputs  (S,N,F)   [model input]
      xr_c, xh_c, xd_c  — CLEAN  inputs  (S,N,F)   [BackNet targets]
      y                  — CLEAN  labels  (P,N)      [prediction target]

    Window layout (all seq_len=12 steps wide):
      X_RecN  : [t - 12           : t          ]  offset=0
      X_HourN : [t - 12 - oh      : t - oh     ]  offset=12
      X_DayN  : [t - 12 - od      : t - od     ]  offset=288
      y       : [t                : t + 12     ]
    """
    def __init__(self, data, cfg, start, end):
        self.data = torch.from_numpy(data)
        self.S    = cfg.seq_len
        self.P    = cfg.pred_len
        self.fi   = cfg.feature_idx
        self.oh   = cfg.offset_hour
        self.od   = cfg.offset_day
        self.nm   = cfg.noise_mean
        self.ns   = cfg.noise_std
        # earliest valid t: need day-ago window + seq_len
        min_t     = cfg.offset_day + cfg.seq_len
        self.idx  = list(range(start + min_t, end - cfg.pred_len + 1))

    def __len__(self): return len(self.idx)

    def _noisy(self, x):
        return x + torch.randn_like(x) * self.ns + self.nm

    def __getitem__(self, i):
        t  = self.idx[i]
        S, P, fi = self.S, self.P, self.fi

        xr_c = self.data[t-S          : t]                    # (S,N,F)
        xh_c = self.data[t-S-self.oh  : t-self.oh]            # (S,N,F)
        xd_c = self.data[t-S-self.od  : t-self.od]            # (S,N,F)
        y    = self.data[t             : t+P, :, fi]           # (P,N)

        return self._noisy(xr_c), self._noisy(xh_c), self._noisy(xd_c),                xr_c, xh_c, xd_c, y


def build_loaders(cfg):
    data, mean, std, A = load_pems08(cfg)
    T  = len(data)
    t1 = int(T * cfg.train_ratio)
    t2 = int(T * (cfg.train_ratio + cfg.val_ratio))
    kw = dict(batch_size=cfg.batch_size, num_workers=2,
               pin_memory=True, drop_last=False)
    dl_tr = DataLoader(MDGRTNDataset(data,cfg,0, t1), shuffle=True,  **kw)
    dl_va = DataLoader(MDGRTNDataset(data,cfg,t1,t2), shuffle=False, **kw)
    dl_te = DataLoader(MDGRTNDataset(data,cfg,t2,T),  shuffle=False, **kw)
    print(f'Train : {len(dl_tr.dataset)} samples  ({len(dl_tr)} batches)')
    print(f'Val   : {len(dl_va.dataset)} samples')
    print(f'Test  : {len(dl_te.dataset)} samples')
    return dl_tr, dl_va, dl_te, mean, std, A

print('Dataset ready.')

In [ ]:
# ══════════════════════════════════════════════════
# DIFFUSION SCHEDULE
# Forward  (Eq.2): X_{t+1} = √(1-β_t)·X_t + √β_t·ε
# Backward (Eq.3): X_t = (X_{t+1} - β_t/√(1-β_t)·ε_θ) / √(1-β_t)
# ══════════════════════════════════════════════════

def make_betas(T, b0, b1):
    """Linear beta schedule."""
    return torch.linspace(b0, b1, T)   # (T,)

# Pre-compute schedule buffers (used in BackNet backward chain)
_betas      = make_betas(cfg.T_diff, cfg.beta_start, cfg.beta_end)
_alphas     = 1.0 - _betas
_alpha_bars = torch.cumprod(_alphas, dim=0)

print('Beta schedule:')
print(f'  β_0={_betas[0]:.4f}  β_T={_betas[-1]:.4f}')
print(f'  ᾱ_0={_alpha_bars[0]:.4f}  ᾱ_T={_alpha_bars[-1]:.4f}')

In [ ]:
# ══════════════════════════════════════════════════
# U-NET BACKNET  (paper Ref[50] + Section IV-A)
# Estimates noise ε_θ(X_{t+1}, t)
# Architecture: Encoder → Bottleneck → Decoder
#               with skip connections + timestep cond
# ══════════════════════════════════════════════════

class SinTimeEmb(nn.Module):
    """Sinusoidal timestep embedding → projected to dim."""
    def __init__(self, dim):
        super().__init__()
        self.dim  = dim
        self.proj = nn.Sequential(
            nn.Linear(dim, dim*2), nn.SiLU(),
            nn.Linear(dim*2, dim))

    def forward(self, t):                    # t: (B,) int64
        half = self.dim // 2
        f = torch.exp(
            -math.log(10000) *
            torch.arange(half, device=t.device, dtype=torch.float32)
            / max(half-1, 1))
        x = t.float().unsqueeze(1) * f.unsqueeze(0)   # (B, half)
        x = torch.cat([x.sin(), x.cos()], dim=1)      # (B, dim)
        return self.proj(x)                            # (B, dim)


class ResBlock(nn.Module):
    """Residual Linear block with timestep conditioning."""
    def __init__(self, in_c, out_c, t_dim):
        super().__init__()
        self.n1  = nn.LayerNorm(in_c)
        self.l1  = nn.Linear(in_c, out_c)
        self.tp  = nn.Linear(t_dim, out_c)
        self.n2  = nn.LayerNorm(out_c)
        self.l2  = nn.Linear(out_c, out_c)
        self.res = nn.Linear(in_c, out_c) if in_c != out_c else nn.Identity()

    def forward(self, x, te):
        # x: (B,S,N,C)   te: (B,t_dim)
        h = F.silu(self.l1(self.n1(x)))
        h = h + self.tp(te)[:,None,None,:]    # broadcast over S,N
        h = self.l2(F.silu(self.n2(h)))
        return h + self.res(x)


class UNetBackNet(nn.Module):
    """
    U-Net noise estimator  ε_θ(X_{t+1}, t).
    Input  : (B,S,N,F)  noisy  +  t (B,) timestep
    Output : (B,S,N,F)  estimated noise
    """
    def __init__(self, in_f, hid=64, t_dim=64):
        super().__init__()
        self.temb = SinTimeEmb(t_dim)
        # Encoder
        self.e1   = ResBlock(in_f,  hid,   t_dim)
        self.e2   = ResBlock(hid,   hid*2, t_dim)
        # Bottleneck
        self.bot  = ResBlock(hid*2, hid*2, t_dim)
        # Decoder  (cat with skip → doubled channels)
        self.d2   = ResBlock(hid*4, hid,   t_dim)
        self.d1   = ResBlock(hid*2, hid,   t_dim)
        # Output head
        self.out  = nn.Linear(hid, in_f)

    def forward(self, x, t):
        te = self.temb(t)
        e1 = self.e1(x,  te)
        e2 = self.e2(e1, te)
        b  = self.bot(e2,te)
        d2 = self.d2(torch.cat([b,  e2], dim=-1), te)
        d1 = self.d1(torch.cat([d2, e1], dim=-1), te)
        return self.out(d1)

print('U-Net BackNet defined  (Encoder→Bottleneck→Decoder + skip + timestep).')

In [ ]:
# ══════════════════════════════════════════════════
# MD MODULE  (paper Eq.3-4)
# 3 separate BackNets, one per period.
# BackNet_k runs the full backward chain T→0.
# X̂_k = BackNet_k(X_k)
# ══════════════════════════════════════════════════

class MDModule(nn.Module):
    def __init__(self, in_f, hid, T_diff, betas):
        super().__init__()
        self.bn_rec  = UNetBackNet(in_f, hid)   # BackNet_1  (Rec)
        self.bn_hour = UNetBackNet(in_f, hid)   # BackNet_2  (Hour)
        self.bn_day  = UNetBackNet(in_f, hid)   # BackNet_3  (Day)
        self.T = T_diff
        self.register_buffer('betas', betas)    # (T,)

    def _step(self, bn, x, t_idx):
        """One backward step — paper Eq.3."""
        b   = self.betas[t_idx]
        t_v = torch.full((x.shape[0],), t_idx,
                         device=x.device, dtype=torch.long)
        eps = bn(x, t_v)                        # estimated noise
        return (x - b / (1-b).sqrt() * eps) / (1-b).sqrt()

    def _denoise(self, bn, x):
        """Full backward chain  T → 0."""
        for t in reversed(range(self.T)):
            x = self._step(bn, x, t)
        return x

    def forward(self, xr_n, xh_n, xd_n):
        """Noisy (B,S,N,F) → clean (B,S,N,F) for each period."""
        return (self._denoise(self.bn_rec,  xr_n),
                self._denoise(self.bn_hour, xh_n),
                self._denoise(self.bn_day,  xd_n))

print('MD Module defined  (3 × UNetBackNet, full backward chain).')

In [ ]:
# ══════════════════════════════════════════════════
# MAF MODULE  (paper Eq.5-9 + architecture figure)
#
# Figure: per period → 3 Linear (Q,K,V) → Dot product
#         3 heads → Concat + FC → X_F1
#
# Eq.5 : Q_k = W_Q_k · X̂_k
# Eq.6 : K_k = W_K_k · X̂_k
# Eq.7 : V_k = W_V_k · X̂_k
# Eq.8 : Attn_k = softmax(Q_k K_k^T / √d_j) · V_k
# Eq.9 : X_F1 = Concat(head_Rec, head_Hour, head_Day) · W_MH
#
# Attention computed over N nodes (spatial)
# ══════════════════════════════════════════════════

class MAFModule(nn.Module):
    def __init__(self, in_f, d_model, n_heads):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_j = d_model // n_heads          # scaling factor

        # Independent Q,K,V for each of 3 periods
        self.W_Q = nn.ModuleList([nn.Linear(in_f, d_model, bias=False) for _ in range(3)])
        self.W_K = nn.ModuleList([nn.Linear(in_f, d_model, bias=False) for _ in range(3)])
        self.W_V = nn.ModuleList([nn.Linear(in_f, d_model, bias=False) for _ in range(3)])

        # Concat + FC  (Eq.9, W_MH)
        self.W_MH = nn.Linear(d_model * 3, d_model)

    def _dot_attn(self, Q, K, V):
        """Scaled dot-product over N nodes  (Eq.8)."""
        # Q,K,V : (B*S, N, d_model)
        sc  = torch.bmm(Q, K.transpose(1,2)) / math.sqrt(self.d_j)
        att = F.softmax(sc, dim=-1)            # (B*S, N, N)
        return torch.bmm(att, V)               # (B*S, N, d_model)

    def forward(self, xr, xh, xd):
        """xr/xh/xd : (B,S,N,F)  →  X_F1 : (B,S,N,d_model)"""
        B, S, N, _ = xr.shape
        heads = []
        for i, x in enumerate([xr, xh, xd]):
            f = x.reshape(B*S, N, -1)          # (B*S, N, F)
            Q = self.W_Q[i](f)                 # (B*S, N, d)
            K = self.W_K[i](f)
            V = self.W_V[i](f)
            h = self._dot_attn(Q, K, V)        # (B*S, N, d)
            heads.append(h.reshape(B, S, N,-1))

        cat   = torch.cat(heads, dim=-1)       # (B, S, N, 3*d)
        X_F1  = self.W_MH(cat)                # (B, S, N, d)
        return X_F1

print('MAF Module defined  (per-period Q,K,V → dot-product → Concat+FC).')

In [ ]:
class MDAFModule(nn.Module):
    """MDAF = MD + MAF  (paper Section IV-A)."""
    def __init__(self, in_f, d_model, n_heads, T_diff, betas):
        super().__init__()
        self.md  = MDModule(in_f, 64, T_diff, betas)
        self.maf = MAFModule(in_f, d_model, n_heads)

    def forward(self, xr_n, xh_n, xd_n):
        xr_c, xh_c, xd_c = self.md(xr_n, xh_n, xd_n)    # denoise
        return self.maf(xr_c, xh_c, xd_c)                 # fuse → X_F1

print('MDAF Module defined.')

In [ ]:
# ══════════════════════════════════════════════════
# MGRC MODULE  (paper Eq.10-14 + figure)
#
# Eq.10: A_dyna = softmax(ReLU(E1 @ E2^T))
#        E1, E2 ∈ R^{N×1}
# Eq.11: A_dist = Gaussian kernel (loaded from npz)
# Eq.12: A_F = ReLU(Conv2D(Concat(A_dyna, A_dist)))
# Eq.13: X'  = ReLU(A_F @ X @ W_GCN)
# Eq.14: GRU gates → h_t
# ══════════════════════════════════════════════════

class MultiGraphFusion(nn.Module):
    """A_dyna + A_dist → A_F  (paper Eq.10, 12)."""
    def __init__(self, N):
        super().__init__()
        self.E1     = nn.Parameter(torch.randn(N,1) * 0.01)
        self.E2     = nn.Parameter(torch.randn(N,1) * 0.01)
        # Concat+2DConv: 2 channels → 1 channel  (paper Eq.12, figure)
        self.conv2d = nn.Conv2d(2, 1, kernel_size=1)

    def forward(self, A_dist):
        # Eq.10
        A_dyna  = F.softmax(F.relu(self.E1 @ self.E2.T), dim=-1)  # (N,N)
        # Eq.12
        stack   = torch.stack([A_dyna, A_dist], dim=0).unsqueeze(0)# (1,2,N,N)
        A_F     = F.relu(self.conv2d(stack)).squeeze()              # (N,N)
        # row-normalise
        return A_F / (A_F.sum(-1, keepdim=True) + 1e-8)


class GCN_GRU_Layer(nn.Module):
    """Single GCN→GRU layer  (paper Eq.13-14, figure: GCN←GRU pair)."""
    def __init__(self, in_d, hid_d):
        super().__init__()
        self.W_GCN = nn.Linear(in_d, hid_d, bias=False)
        self.gru   = nn.GRUCell(hid_d, hid_d)

    def forward(self, x_seq, A_F):
        # x_seq: (B,S,N,in_d)
        B, S, N, _ = x_seq.shape
        h = torch.zeros(B*N, self.gru.hidden_size, device=x_seq.device)
        outs = []
        for t in range(S):
            # Eq.13: X' = ReLU(A_F @ X_t @ W_GCN)
            agg  = torch.einsum('nm,bmd->bnd', A_F, x_seq[:,t])
            xp   = F.relu(self.W_GCN(agg))           # (B,N,hid)
            # Eq.14: GRU update
            h    = self.gru(xp.reshape(B*N,-1), h)
            outs.append(h.reshape(B,N,-1))
        return torch.stack(outs, dim=1)               # (B,S,N,hid)


class MGRCModule(nn.Module):
    """6 stacked GCN+GRU layers with dual-graph adjacency."""
    def __init__(self, in_d, hid_d, N, n_layers=6):
        super().__init__()
        self.fusion = MultiGraphFusion(N)
        dims = [in_d] + [hid_d]*n_layers
        self.layers = nn.ModuleList([
            GCN_GRU_Layer(dims[i], dims[i+1]) for i in range(n_layers)])

    def forward(self, X_F1, A_dist):
        A_F = self.fusion(A_dist)
        x   = X_F1
        for layer in self.layers:
            x = layer(x, A_F)
        return x    # X_F2: (B,S,N,hid)

print('MGRC Module defined  (6×GCN+GRU, A_dyna+A_dist→Conv→A_F).')

In [ ]:
# ══════════════════════════════════════════════════
# STFORMER  (paper Eq.15-26 + figure)
#
# Spatial Transformer (per layer l):
#   Eq.15: X_S1 = X_ST + A·W_SPE          (adj-weighted spatial pos)
#   Eq.16: X_S2 = SMHA(X_S1)
#   Eq.17: X_S3 = LN(X_S2 + X_S1)
#   Eq.18: X_S4 = FF(X_S3)
#   Eq.19: Y_S  = LN(X_S4 + FF(X_S4))
#
# Temporal Transformer (per layer l):
#   Eq.20: X_T1 = LN(X_ST_prev + Y_S)     ← spatial residual!
#   Eq.21: X_T2 = X_T1 + W_h·E_h + W_d·E_d + W_w·E_w
#   Eq.22: X_T3 = TMHA(X_T2)
#   Eq.23: X_T3 = LN(X_T3 + X_T2)
#   Eq.24: X_T4 = FF(X_T3)
#   Eq.25: X_ST = LN(X_T4 + FF(X_T4))
# ══════════════════════════════════════════════════

class TemporalPosEmb(nn.Module):
    """Paper Eq.21 — 3-component temporal encoding."""
    def __init__(self, N, S):
        super().__init__()
        self.W_h = nn.Parameter(torch.randn(N,1)*0.01)
        self.W_d = nn.Parameter(torch.randn(N,1)*0.01)
        self.W_w = nn.Parameter(torch.randn(N,1)*0.01)
        idx = torch.arange(1, S+1, dtype=torch.float32)
        self.register_buffer('E_h', idx % 60)   # hourly  ∈[1,60]
        self.register_buffer('E_d', idx % 24)   # daily   ∈[1,24]
        self.register_buffer('E_w', idx % 7)    # weekly  ∈[1,7]

    def forward(self, x):
        # x: (B,S,N,d)
        # W: (N,1)  E: (S,)  → outer (N,S) → permute (S,N) → (1,S,N,1)
        def _emb(W, E):
            return (W * E.unsqueeze(0)).permute(1,0).unsqueeze(0).unsqueeze(-1)
        return x + _emb(self.W_h, self.E_h)                  + _emb(self.W_d, self.E_d)                  + _emb(self.W_w, self.E_w)


class STFormerLayer(nn.Module):
    """One STFormer layer = Spatial + Temporal transformer."""
    def __init__(self, d, nh, N, S, dr=0.1):
        super().__init__()
        assert d % nh == 0, f"d={d} not divisible by nh={nh}"

        # ── Spatial ──
        self.W_SPE = nn.Parameter(torch.randn(d,d)*0.01)  # Eq.15
        self.SMHA  = nn.MultiheadAttention(d, nh, dropout=dr, batch_first=True)
        self.LN_s1 = nn.LayerNorm(d)
        self.FF_s  = nn.Sequential(nn.Linear(d,d*4), nn.ReLU(), nn.Linear(d*4,d))
        self.LN_s2 = nn.LayerNorm(d)

        # ── Temporal ──
        self.LN_t0 = nn.LayerNorm(d)                       # Eq.20
        self.TPE   = TemporalPosEmb(N, S)                  # Eq.21
        self.TMHA  = nn.MultiheadAttention(d, nh, dropout=dr, batch_first=True)
        self.LN_t1 = nn.LayerNorm(d)
        self.FF_t  = nn.Sequential(nn.Linear(d,d*4), nn.ReLU(), nn.Linear(d*4,d))
        self.LN_t2 = nn.LayerNorm(d)
        self.drop  = nn.Dropout(dr)

    def _spatial(self, X, A):
        B,S,N,d = X.shape
        # Eq.15: spatial pos embedding weighted by adjacency A
        spe  = torch.einsum('nm,md->nd', A, self.W_SPE)   # (N,d)
        X_S1 = X + spe[None,None]                          # (B,S,N,d)
        # Eq.16-17
        f    = X_S1.reshape(B*S,N,d)
        h,_  = self.SMHA(f,f,f)
        X_S3 = self.LN_s1(f + self.drop(h))
        # Eq.18-19
        X_S4 = self.FF_s(X_S3)
        Y_S  = self.LN_s2(X_S3 + self.drop(X_S4))
        return Y_S.reshape(B,S,N,d)

    def _temporal(self, X_prev, Y_S):
        B,S,N,d = X_prev.shape
        # Eq.20: residual from spatial branch
        X_T1 = self.LN_t0(X_prev + Y_S)
        # Eq.21: temporal pos encoding
        X_T2 = self.TPE(X_T1)
        # Eq.22-23
        f    = X_T2.permute(0,2,1,3).reshape(B*N,S,d)
        h,_  = self.TMHA(f,f,f)
        X_T3 = self.LN_t1(f + self.drop(h))
        # Eq.24-25
        X_T4 = self.FF_t(X_T3)
        X_ST = self.LN_t2(X_T3 + self.drop(X_T4))
        return X_ST.reshape(B,N,S,d).permute(0,2,1,3)     # (B,S,N,d)

    def forward(self, X_ST, A):
        Y_S  = self._spatial(X_ST, A)
        X_ST = self._temporal(X_ST, Y_S)
        return X_ST


class STFormerModule(nn.Module):
    """L=3 STFormer layers → FC output  (Eq.26)."""
    def __init__(self, d, nh, N, S, P, L=3, dr=0.1):
        super().__init__()
        self.layers = nn.ModuleList(
            [STFormerLayer(d,nh,N,S,dr) for _ in range(L)])
        self.fc_out = nn.Linear(d*S, P)   # Eq.26: Y_Feat = FC(Y_ST)

    def forward(self, X_F2, A):
        x = X_F2
        for layer in self.layers:
            x = layer(x, A)
        B,S,N,d = x.shape
        out = self.fc_out(x.permute(0,2,1,3).reshape(B,N,S*d))  # (B,N,P)
        return out.permute(0,2,1)                                  # (B,P,N)

print('STFormer defined  (L=3, Eq.15-26, adj-weighted spatial pos, 3-part temporal pos).')

In [ ]:
# ══════════════════════════════════════════════════
# FULL MD-GRTN
# ══════════════════════════════════════════════════

class MDGRTN(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        N, F, d = cfg.num_nodes, cfg.in_features, cfg.d_model
        betas   = make_betas(cfg.T_diff, cfg.beta_start, cfg.beta_end)

        self.mdaf     = MDAFModule(F, d, cfg.n_heads, cfg.T_diff, betas)
        self.mgrc     = MGRCModule(d, d, N, cfg.mgrc_layers)
        self.stformer = STFormerModule(d, cfg.n_heads, N,
                                       cfg.seq_len, cfg.pred_len,
                                       cfg.num_layers, cfg.dropout)

    def forward(self, xr_n, xh_n, xd_n, A):
        X_F1 = self.mdaf(xr_n, xh_n, xd_n)   # (B,S,N,d)
        X_F2 = self.mgrc(X_F1, A)             # (B,S,N,d)
        return self.stformer(X_F2, A)         # (B,P,N)


model = MDGRTN(cfg).to(device)
total = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Parameters : {total:,}')

# ── Sanity check ──
with torch.no_grad():
    dummy = torch.randn(2, cfg.seq_len, cfg.num_nodes, cfg.in_features).to(device)
    adj   = torch.eye(cfg.num_nodes).to(device)
    out   = model(dummy, dummy, dummy, adj)
print(f'Output shape: {out.shape}   ✓  (expected [2, {cfg.pred_len}, {cfg.num_nodes}])')

In [ ]:
# ══════════════════════════════════════════════════
# METRICS  (paper Eq.28-30)
# ══════════════════════════════════════════════════

def MAE(p, y):
    """Eq.29"""
    return torch.abs(p - y).mean()

def RMSE(p, y):
    """Eq.28"""
    return ((p-y)**2).mean().sqrt()

def MAPE(p, y, thresh=10.0):
    """Eq.30 — mask low-flow to avoid ÷0"""
    mask = y.abs() > thresh
    if not mask.any(): return torch.tensor(0.0, device=p.device)
    return (torch.abs((p[mask]-y[mask]) / (y[mask].abs()+1.0))).mean() * 100

def huber(p, y, d=1.0):
    """Eq.27"""
    return F.huber_loss(p, y, delta=d)

def denorm(x, mf, sf):
    """Denormalise predictions back to real vehicle counts."""
    return x * sf[None,None,:] + mf[None,None,:]

print('Metrics defined.')

In [ ]:
# ── Mount Drive (uncomment if using Drive) ──
# from google.colab import drive
# drive.mount('/content/drive')
# cfg.data_path = '/content/drive/MyDrive/PEMS08.npz'

dl_tr, dl_va, dl_te, mean_np, std_np, A_np = build_loaders(cfg)

mean_flow = torch.from_numpy(mean_np[0,:,cfg.feature_idx]).to(device)  # (N,)
std_flow  = torch.from_numpy(std_np [0,:,cfg.feature_idx]).to(device)
A_dist    = torch.from_numpy(A_np).to(device)

print(f'A_dist shape : {A_dist.shape}')
print(f'mean_flow    : min={mean_flow.min():.2f}  max={mean_flow.max():.2f}')

In [ ]:
# ══════════════════════════════════════════════════
# PHASE 1 — PRE-TRAIN MD MODULE
# Algorithm 1 lines 1-6:
#   for epoch in max_epoch:
#     Ĥ_k = BackNet_k(X_k)
#     Loss = MSE(Ĥ_k, X̂_k)   ← clean target
#     update θ_MD
#   freeze θ_MD
# ══════════════════════════════════════════════════

def pretrain_md(model, loader, cfg, device):
    params = list(model.mdaf.md.parameters())
    opt    = torch.optim.AdamW(params, lr=cfg.lr, weight_decay=cfg.weight_decay)
    losses = []
    it     = 0
    model.train()
    print(f'Pre-training MD ({cfg.pretrain_iters} iters)...')

    while it < cfg.pretrain_iters:
        for xr_n,xh_n,xd_n,xr_c,xh_c,xd_c,_ in loader:
            xr_n=xr_n.to(device); xh_n=xh_n.to(device); xd_n=xd_n.to(device)
            xr_c=xr_c.to(device); xh_c=xh_c.to(device); xd_c=xd_c.to(device)

            pr,ph,pd = model.mdaf.md(xr_n, xh_n, xd_n)
            loss = (F.mse_loss(pr,xr_c) +
                    F.mse_loss(ph,xh_c) +
                    F.mse_loss(pd,xd_c)) / 3

            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(params, 5.0)
            opt.step()
            losses.append(loss.item()); it += 1

            if it % 50 == 0:
                print(f'  iter {it:4d}/{cfg.pretrain_iters} | MSE = {loss.item():.5f}')
            if it >= cfg.pretrain_iters: break

    # Freeze MD weights (Algorithm 1: load in main training, not updated)
    for p in model.mdaf.md.parameters():
        p.requires_grad_(False)
    torch.save(model.mdaf.md.state_dict(), 'md_pretrained.pt')
    print('Pre-training done ✓   MD module frozen.')
    return losses


pretrain_losses = pretrain_md(model, dl_tr, cfg, device)

plt.figure(figsize=(7,3))
plt.plot(pretrain_losses)
plt.title('MD Pre-training Loss (MSE)')
plt.xlabel('Iteration'); plt.ylabel('MSE Loss')
plt.tight_layout(); plt.savefig('pretrain_loss.png',dpi=120); plt.show()

In [ ]:
# ══════════════════════════════════════════════════
# PHASE 2 — MAIN TRAINING
# Algorithm 1 lines 7-17:
#   update θ_MAF, θ_MGRC, θ_ST, θ_TT
#   Loss = Huber(Y_R, Y_P)  [Eq.27]
# ══════════════════════════════════════════════════

def run_epoch(model, loader, opt, A, device, cfg, train=True):
    model.train() if train else model.eval()
    tot, n = 0.0, 0
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for xr_n,xh_n,xd_n,_,_,_,y in loader:
            xr_n=xr_n.to(device); xh_n=xh_n.to(device)
            xd_n=xd_n.to(device); y=y.to(device)
            pred = model(xr_n, xh_n, xd_n, A)
            loss = huber(pred, y, cfg.delta_h)
            if train:
                opt.zero_grad(); loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
                opt.step()
            tot += loss.item(); n += 1
    return tot / n

@torch.no_grad()
def evaluate(model, loader, A, device, mf, sf):
    model.eval()
    maes,rmses,mapes = [],[],[]
    for xr_n,xh_n,xd_n,_,_,_,y in loader:
        xr_n=xr_n.to(device); xh_n=xh_n.to(device)
        xd_n=xd_n.to(device); y=y.to(device)
        pred = model(xr_n, xh_n, xd_n, A)
        pd   = denorm(pred,mf,sf); yd = denorm(y,mf,sf)
        maes.append(MAE(pd,yd).item())
        rmses.append(RMSE(pd,yd).item())
        mapes.append(MAPE(pd,yd).item())
    return np.mean(maes), np.mean(rmses), np.mean(mapes)


# Only non-frozen params (MD is frozen)
main_params = [p for p in model.parameters() if p.requires_grad]
print(f'Trainable params (excl. MD): {sum(p.numel() for p in main_params):,}')
opt = torch.optim.AdamW(main_params, lr=cfg.lr, weight_decay=cfg.weight_decay)

history = {'loss':[],'val_mae':[],'val_rmse':[],'val_mape':[]}
best_mae = float('inf')
total_iters = 0
iters_per_ep = len(dl_tr)
max_ep = max(1, cfg.max_iters // iters_per_ep)

print(f'\nMain training | ~{max_ep} epochs | max {cfg.max_iters} iters')
print(f'Paper target  → MAE=13.114 | RMSE=22.623 | MAPE=8.471%')
print('='*68)

for ep in range(1, max_ep+1):
    loss = run_epoch(model, dl_tr, opt, A_dist, device, cfg, train=True)
    vm,vr,vp = evaluate(model, dl_va, A_dist, device, mean_flow, std_flow)
    total_iters += iters_per_ep

    history['loss'].append(loss)
    history['val_mae'].append(vm)
    history['val_rmse'].append(vr)
    history['val_mape'].append(vp)

    tag = ' ← best ✓' if vm < best_mae else ''
    print(f'Ep {ep:03d} | ~{total_iters} iters | Loss={loss:.4f} | '
          f'MAE={vm:.3f}  RMSE={vr:.3f}  MAPE={vp:.2f}%{tag}')

    if vm < best_mae:
        best_mae = vm
        torch.save(model.state_dict(), 'mdgrtn_best.pt')

    if total_iters >= cfg.max_iters:
        print('Max iterations reached.'); break

In [ ]:
fig, axes = plt.subplots(1,3,figsize=(16,4))
axes[0].plot(history['loss'], color='steelblue', lw=2)
axes[0].set_title('Training Loss (Huber)'); axes[0].set_xlabel('Epoch')

axes[1].plot(history['val_mae'], color='orange', lw=2, label='Val MAE')
axes[1].axhline(13.114, color='red', ls='--', lw=1.5, label='Paper 13.114')
axes[1].set_title('Validation MAE'); axes[1].legend()

axes[2].plot(history['val_rmse'], color='green', lw=2, label='Val RMSE')
axes[2].axhline(22.623, color='red', ls='--', lw=1.5, label='Paper 22.623')
axes[2].set_title('Validation RMSE'); axes[2].legend()

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150); plt.show()

In [ ]:
model.load_state_dict(torch.load('mdgrtn_best.pt', map_location=device))
tm, tr, tp = evaluate(model, dl_te, A_dist, device, mean_flow, std_flow)

print('\n' + '='*60)
print('  TEST RESULTS  —  PEMS08  (all 12 steps averaged)')
print('='*60)
print(f'  MAE  : {tm:.3f}    paper: 13.114   Δ = {tm-13.114:+.3f}')
print(f'  RMSE : {tr:.3f}    paper: 22.623   Δ = {tr-22.623:+.3f}')
print(f'  MAPE : {tp:.2f}%   paper:  8.471%  Δ = {tp-8.471:+.2f}%')
print('='*60)

In [ ]:
@torch.no_grad()
def horizon_eval(model, loader, A, device, mf, sf):
    model.eval()
    buf = {h:{'mae':[],'rmse':[],'mape':[]} for h in [2,5,11]}
    for xr_n,xh_n,xd_n,_,_,_,y in loader:
        xr_n=xr_n.to(device); xh_n=xh_n.to(device)
        xd_n=xd_n.to(device); y=y.to(device)
        pred = model(xr_n,xh_n,xd_n,A)
        pd   = denorm(pred,mf,sf); yd = denorm(y,mf,sf)
        for h in buf:
            buf[h]['mae'].append(MAE(pd[:,h,:],  yd[:,h,:]).item())
            buf[h]['rmse'].append(RMSE(pd[:,h,:], yd[:,h,:]).item())
            buf[h]['mape'].append(MAPE(pd[:,h,:], yd[:,h,:]).item())

    print(f'{"Horizon":>14} | {"MAE":>8} | {"RMSE":>8} | {"MAPE":>9}')
    print('-'*52)
    for h,lbl in zip([2,5,11],['3-step (15min)','6-step (30min)','12-step (60min)']):
        m = {k:np.mean(v) for k,v in buf[h].items()}
        print(f'{lbl:>14} | {m["mae"]:>8.3f} | {m["rmse"]:>8.3f} | {m["mape"]:>8.2f}%')

horizon_eval(model, dl_te, A_dist, device, mean_flow, std_flow)